# Hunyuan3D-2.1-macOS Fork — CUDA Verification

This notebook verifies that the fork's changes (MLX imports guarded by `try/except`, `HUNYUAN_DISABLE_MLX` logic, `device_utils` auto-detection) don't break CUDA.

On CUDA, MLX setup is skipped entirely (`if str(self.device) != 'cuda'`).

**Requirements:** Colab GPU runtime (T4/A100)

In [ ]:
# Cell 1: Clone + install
!git clone https://github.com/noahcorona/Hunyuan3D-2.1-macOS.git
%cd Hunyuan3D-2.1-macOS

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -r requirements.txt

# Build rasterizer
%cd hy3dpaint/custom_rasterizer
!pip install -e .
%cd ../..

# Build mesh painter
%cd hy3dpaint/DifferentiableRenderer
!bash compile_mesh_painter.sh
%cd ../..

# Download RealESRGAN weights
!mkdir -p ckpt
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P ckpt

In [ ]:
# Cell 2: Shape generation (CUDA)
import sys
import time
import torch

sys.path.insert(0, './hy3dshape')
from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

# Load shape pipeline — on CUDA, MLX setup is skipped
t0 = time.time()
shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2.1',
    device='cuda',
    dtype=torch.float16,
)
print(f"Shape pipeline loaded in {time.time() - t0:.1f}s")

# Generate mesh
t0 = time.time()
mesh = shape_pipeline('assets/test_inputs/aloe_plant.png')[0]
shape_time = time.time() - t0
print(f"Shape generation: {shape_time:.1f}s")

# Save mesh for paint step
mesh.export('test_output.glb')
print(f"Mesh: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces")

In [ ]:
# Cell 3: Paint generation (CUDA)
sys.path.insert(0, './hy3dpaint')
from textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig

t0 = time.time()
paint_config = Hunyuan3DPaintConfig(max_num_view=6, resolution=512)
paint_pipeline = Hunyuan3DPaintPipeline(paint_config)
print(f"Paint pipeline loaded in {time.time() - t0:.1f}s")

t0 = time.time()
textured_mesh = paint_pipeline(
    'test_output.glb',
    image_path='assets/test_inputs/aloe_plant.png',
)
paint_time = time.time() - t0
print(f"Paint generation: {paint_time:.1f}s")

In [ ]:
# Cell 4: Results summary
print("=" * 50)
print("CUDA Verification Results")
print("=" * 50)
print(f"Shape generation: {shape_time:.1f}s")
print(f"Paint generation: {paint_time:.1f}s")
print(f"Total: {shape_time + paint_time:.1f}s")
print()
print("Fork changes verified on CUDA — MLX codepaths skipped correctly.")